In [1]:
# ============================================================
# MODEL 2 — SEASONALITY SPECIALIST
# Clean rebuild using only Seasonality history
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import zipfile
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
!pip install duckdb -q
import duckdb

con = duckdb.connect()

# Extract transactions
with zipfile.ZipFile('/content/drive/MyDrive/transactions_features.zip', 'r') as z:
    z.extractall('/content/')

# Load files
train = pd.read_csv('/content/drive/MyDrive/Train.csv')
test = pd.read_csv('/content/drive/MyDrive/Test.csv')
financials = pd.read_parquet('/content/drive/MyDrive/financials_features.parquet')
demographics = pd.read_parquet('/content/drive/MyDrive/demographics_clean.parquet')

print("✅ Data loaded!")
print(f"Train: {train.shape} | Test: {test.shape}")

Mounted at /content/drive
✅ Data loaded!
Train: (8360, 2) | Test: (3584, 1)


In [2]:
# ============================================================
# MODEL 2 — SEASONALITY SPECIALIST
# Focus: Same period last year + monthly patterns
# ============================================================

seasonality_features = con.execute("""
    SELECT
        UniqueID,

        -- Same period last year (THE GOLDEN FEATURE)
        SUM(CASE WHEN TransactionDate >= DATE '2014-11-01'
                  AND TransactionDate < DATE '2015-02-01'
                  THEN 1 ELSE 0 END) as txn_same_period_lastyear,

        -- Same period last year monthly breakdown
        SUM(CASE WHEN TransactionDate >= DATE '2014-11-01'
                  AND TransactionDate < DATE '2014-12-01'
                  THEN 1 ELSE 0 END) as txn_nov_lastyear,
        SUM(CASE WHEN TransactionDate >= DATE '2014-12-01'
                  AND TransactionDate < DATE '2015-01-01'
                  THEN 1 ELSE 0 END) as txn_dec_lastyear,
        SUM(CASE WHEN TransactionDate >= DATE '2015-01-01'
                  AND TransactionDate < DATE '2015-02-01'
                  THEN 1 ELSE 0 END) as txn_jan_lastyear,

        -- Two years ago same period
        SUM(CASE WHEN TransactionDate >= DATE '2013-11-01'
                  AND TransactionDate < DATE '2014-02-01'
                  THEN 1 ELSE 0 END) as txn_same_period_2yrsago,

        -- Holiday season patterns
        SUM(CASE WHEN MONTH(TransactionDate) = 11
                  THEN 1 ELSE 0 END) as all_november_txns,
        SUM(CASE WHEN MONTH(TransactionDate) = 12
                  THEN 1 ELSE 0 END) as all_december_txns,
        SUM(CASE WHEN MONTH(TransactionDate) = 1
                  THEN 1 ELSE 0 END) as all_january_txns,

        -- Recent months leading to prediction window
        SUM(CASE WHEN MONTH(TransactionDate) = 10
                  AND YEAR(TransactionDate) = 2015
                  THEN 1 ELSE 0 END) as txn_oct_2015,
        SUM(CASE WHEN MONTH(TransactionDate) = 9
                  AND YEAR(TransactionDate) = 2015
                  THEN 1 ELSE 0 END) as txn_sep_2015,
        SUM(CASE WHEN MONTH(TransactionDate) = 8
                  AND YEAR(TransactionDate) = 2015
                  THEN 1 ELSE 0 END) as txn_aug_2015,

        -- October last year (pre-prediction period last year)
        SUM(CASE WHEN MONTH(TransactionDate) = 10
                  AND YEAR(TransactionDate) = 2014
                  THEN 1 ELSE 0 END) as txn_oct_2014,
        SUM(CASE WHEN MONTH(TransactionDate) = 9
                  AND YEAR(TransactionDate) = 2014
                  THEN 1 ELSE 0 END) as txn_sep_2014,

        -- Month end patterns
        SUM(CASE WHEN DAY(TransactionDate) >= 25
                  THEN 1 ELSE 0 END) as month_end_txns,
        SUM(CASE WHEN DAY(TransactionDate) BETWEEN 1 AND 5
                  THEN 1 ELSE 0 END) as month_start_txns,

        -- Current year totals for context
        SUM(CASE WHEN TransactionDate >= DATE '2015-08-01'
                  THEN 1 ELSE 0 END) as txn_last_3m,
        SUM(CASE WHEN TransactionDate >= DATE '2015-05-01'
                  AND TransactionDate < DATE '2015-08-01'
                  THEN 1 ELSE 0 END) as txn_prev_3m,

        -- Active months
        COUNT(DISTINCT DATE_TRUNC('month',
              TransactionDate)) as active_months,

        -- Years of data available
        DATEDIFF('year', MIN(TransactionDate),
                 DATE '2015-10-31') as years_of_history

    FROM '/content/transactions_features.parquet'
    WHERE TransactionTypeDescription != 'Reversals & Adjustments'
    GROUP BY UniqueID
""").df()

# Derived seasonality features
seasonality_features['yoy_growth'] = (
    seasonality_features['txn_same_period_lastyear'] /
    (seasonality_features['txn_same_period_2yrsago'] + 1)
).clip(0, 5)

seasonality_features['recent_vs_lastyear'] = (
    seasonality_features['txn_last_3m'] /
    (seasonality_features['txn_same_period_lastyear'] + 1)
).clip(0, 5)

seasonality_features['holiday_consistency'] = (
    seasonality_features['all_november_txns'] +
    seasonality_features['all_december_txns'] +
    seasonality_features['all_january_txns']
)

seasonality_features['trend_ratio'] = (
    seasonality_features['txn_last_3m'] /
    (seasonality_features['txn_prev_3m'] + 1)
).clip(0, 5)

seasonality_features['month_end_ratio'] = (
    seasonality_features['month_end_txns'] /
    (seasonality_features['month_end_txns'] +
     seasonality_features['month_start_txns'] + 1)
)

# Oct momentum vs last year
seasonality_features['oct_momentum'] = (
    seasonality_features['txn_oct_2015'] /
    (seasonality_features['txn_oct_2014'] + 1)
).clip(0, 5)

print(f"✅ Seasonality features: {seasonality_features.shape}")
print(seasonality_features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Seasonality features: (11944, 26)
                               UniqueID  txn_same_period_lastyear  \
0  f9d8aba5-1818-439e-8627-16ae0d515e50                     367.0   
1  32d8aaf1-ebaa-419b-8462-ccfe2bf82917                     421.0   
2  666cbe05-6bba-4b6d-a07c-ef4567600016                      62.0   
3  6e9e0985-b258-4c1c-95eb-2ac73dae2dc1                     372.0   
4  c90f3a4a-d0e3-4541-a57b-4acda66d13e1                     837.0   

   txn_nov_lastyear  txn_dec_lastyear  txn_jan_lastyear  \
0              81.0             153.0             133.0   
1             120.0             165.0             136.0   
2              14.0              26.0              22.0   
3             125.0             132.0             115.0   
4             232.0             279.0             326.0   

   txn_same_period_2yrsago  all_november_txns  all_december_txns  \
0                      0.0               81.0              153.0   
1                    390.0              251.0             

In [3]:
# ============================================================
# BUILD MASTER — MODEL 2
# ============================================================

master_m2 = train.merge(
    seasonality_features, on='UniqueID', how='left'
)

duplicates = [col for col in master_m2.columns
              if col.endswith('_x') or col.endswith('_y')]
print(f"Duplicates: {duplicates}")
print(f"Master shape: {master_m2.shape}")

drop_cols = ['UniqueID', 'next_3m_txn_count']
feature_cols_m2 = [c for c in master_m2.columns
                   if c not in drop_cols
                   and master_m2[c].dtype != 'datetime64[ns]']

print(f"Features: {len(feature_cols_m2)}")

X_m2 = master_m2[feature_cols_m2].fillna(0)
y_m2 = master_m2['next_3m_txn_count']

Duplicates: []
Master shape: (8360, 27)
Features: 25


In [4]:
# ============================================================
# TRAIN MODEL 2
# ============================================================

X_tr, X_vl, y_tr, y_vl = train_test_split(
    X_m2, y_m2, test_size=0.2, random_state=42
)

model_m2 = XGBRegressor(
    objective='count:poisson',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    random_state=42,
    early_stopping_rounds=50
)

model_m2.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=100)

val_preds_m2 = np.maximum(model_m2.predict(X_vl), 0)
rmsle_m2 = np.sqrt(mean_squared_log_error(y_vl, val_preds_m2))
print(f"\nModel 2 RMSLE:  {rmsle_m2:.4f}")
print(f"Winning RMSLE:  0.4142")
print(f"Difference:     {0.4142 - rmsle_m2:.4f}")

best_n_m2 = model_m2.best_iteration

importance_m2 = pd.DataFrame({
    'feature': feature_cols_m2,
    'importance': model_m2.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 15 features:")
print(importance_m2.head(15))

[0]	validation_0-poisson-nloglik:64.95438
[100]	validation_0-poisson-nloglik:10.85301
[200]	validation_0-poisson-nloglik:9.98688
[261]	validation_0-poisson-nloglik:9.97655

Model 2 RMSLE:  0.4326
Winning RMSLE:  0.4142
Difference:     -0.0184

Top 15 features:
                feature  importance
15          txn_last_3m    0.553058
8          txn_oct_2015    0.228372
16          txn_prev_3m    0.039695
9          txn_sep_2015    0.032646
10         txn_aug_2015    0.015111
7      all_january_txns    0.010517
21  holiday_consistency    0.010087
18     years_of_history    0.008763
12         txn_sep_2014    0.008278
6     all_december_txns    0.007941
14     month_start_txns    0.007109
2      txn_dec_lastyear    0.006839
17        active_months    0.006754
23      month_end_ratio    0.006680
24         oct_momentum    0.006487


In [5]:
# ============================================================
# SUBMIT MODEL 2
# ============================================================

model_m2_final = XGBRegressor(
    objective='count:poisson',
    n_estimators=best_n_m2,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    random_state=42
)

model_m2_final.fit(X_m2, y_m2)
print("✅ Model 2 trained on all data!")

test_m2 = test.merge(seasonality_features, on='UniqueID', how='left')
X_test_m2 = test_m2[feature_cols_m2].fillna(0)

preds_m2 = model_m2_final.predict(X_test_m2)
preds_m2 = np.maximum(preds_m2, 0).round().astype(int)

submission_m2 = pd.DataFrame({
    'UniqueID': test['UniqueID'],
    'next_3m_txn_count': preds_m2
})

submission_m2.to_csv(
    '/content/drive/MyDrive/model2_seasonality_specialist.csv',
    index=False
)

print("✅ Saved model2_seasonality_specialist.csv!")
print(submission_m2['next_3m_txn_count'].describe())

✅ Model 2 trained on all data!
✅ Saved model2_seasonality_specialist.csv!
count    3584.000000
mean      143.417132
std       143.821256
min         5.000000
25%        41.000000
50%       106.000000
75%       193.000000
max      1482.000000
Name: next_3m_txn_count, dtype: float64
